# 00 · Data preparation

**Question.** What data does the repository hold (files, sizes, SHA-256 checksums), how large is the exome BED territory, and what do the truth VCFs contain?

**Reviewer comment.** None.

**Produces** (`results/00_data_preparation/tables/`):
- `inventory_committed.csv`: path, size and SHA-256 of every file in `data/truth/`, `data/reference/` and the committed files of `data/derived/`. It is the same on every checkout, so a reviewer can compare their copy against it.
- `inventory_local.csv`: the same for local-only files: `data/raw/snv/`, `data/raw/indel/` and the `data/derived/*.parquet` caches. Written only if such files exist.
- `inventory_summary.csv`: file count and bytes per folder.
- `bed_territory.csv`: the exome BED territory as the raw sum of interval lengths (overlaps counted twice, as in legacy) and as the merged unique length.
- `truth_census.csv`: the two truth VCFs by FILTER value and variant type.

**Runs without `data/raw/`.** This notebook reads `data/raw/` only to list and hash it (several GB). If a raw folder is absent it says so, skips the raw-dependent parity checks (reported as SKIP, not PASS) and continues. This is a deliberate exception to the rule that a notebook reading `data/raw/` stops when it is missing; see `data/README.md`.

**Territory (D11).** Legacy TMB divides by the raw sum. Both lengths are reported; which one new analysis should use is undecided (`docs/legacy-deviations.md` D11).

**Does not.**
- parse any run VCF (notebook 01);
- build a call set or compute an analysis metric. The legacy TMB and the whole-study metrics are recomputed only as parity checks;
- write the run manifest. Notebook 01 selects the 480 runs from `data/raw/snv/` itself.

Run from `notebooks/`.

## Imports

In [1]:
import sys
from pathlib import Path

try:
    import swb  # noqa: F401
except ImportError:  # swb is not pip-installed: use the repository's src/ (notebooks run from notebooks/)
    sys.path.insert(0, str(Path.cwd().parent / "src"))

import os

import numpy as np
import pandas as pd

from swb import audit, config, io, metrics

## Config

In [2]:
OUT = config.results_dir("00_data_preparation")
STAGE = audit.Stage(OUT / ".staging")

EXPECTED_SNV_RUNS = 480
EXPECTED_INDEL_RUNS = 320
EXPECTED_METADATA_ROWS = 864
EXPECTED_INDEL_METADATA_SHAPE = (320, 13)
TOLERANCE = 1e-12  # recomputed TMB and metrics differ from the legacy CSVs by about 1e-14 or less

TRUTH_INDEL_VCF = config.TRUTH / "hc_bed_filtered_INDELS.recode.vcf"
INDEL_METADATA_CSV = config.REFERENCE / "TestCases_Indels.csv"
LEGACY_CACHES = [config.LEGACY_METADATA, config.LEGACY_FILENAMES, config.LEGACY_FILENAMES_INDELS, config.LEGACY_SETS,
                 config.LEGACY_FILTERING, config.LEGACY_UNION_METRICS, config.LEGACY_TMB]
# Legacy VCF paths are relative to the old drive layout; ours are relative to data/raw/snv and data/raw/indel
LEGACY_SNV_ROOT, LEGACY_INDEL_ROOT = "vcf/TESTCASES_bedded/", "vcf/TESTCASES_bedded_indels/"


def rel(path):
    """POSIX path relative to the repository root."""
    return Path(path).relative_to(config.REPO_ROOT).as_posix()


def is_local(relpath):
    """Files that are not in git (mirrors .gitignore): raw VCFs, and the parquet and *_full_cache* caches."""
    name = relpath.split("/")[-1]
    in_derived = relpath.startswith(rel(config.DERIVED) + "/")
    return relpath.startswith(rel(config.DATA / "raw") + "/") or (
        in_derived and (name.endswith(".parquet") or "_full_cache" in name))

## Load + validate

In [3]:
for p in [config.TRUTH_VCF, TRUTH_INDEL_VCF, config.EXOME_BED, config.METADATA_CSV, INDEL_METADATA_CSV] + LEGACY_CACHES:
    assert p.is_file() and p.stat().st_size > 0, "missing or empty: " + rel(p)

# TestCases.csv (24 columns) and TestCases_Indels.csv (13 columns, no Environment) have different schemas
metadata = pd.read_csv(config.METADATA_CSV)
assert list(metadata.columns) == config.METADATA_COLUMNS, list(metadata.columns)
assert len(metadata) == EXPECTED_METADATA_ROWS, len(metadata)
assert metadata["TestCaseNo"].notna().all() and metadata["TestCaseNo"].map(io.canon).is_unique
assert sorted(metadata["phase"].unique()) == ["done", "notReady"], sorted(metadata["phase"].unique())
indel_metadata = pd.read_csv(INDEL_METADATA_CSV)
assert indel_metadata.shape == EXPECTED_INDEL_METADATA_SHAPE, indel_metadata.shape
assert "Environment" not in indel_metadata.columns

# data/raw is optional here: report what is present and go on
raw_dirs = {"snv": config.RAW_SNV, "indel": config.RAW_INDEL}
raw_present = {name: d.is_dir() for name, d in raw_dirs.items()}
for name, d in raw_dirs.items():
    print("{}: {}".format(rel(d), "present" if raw_present[name] else "ABSENT (not inventoried; raw parity checks skipped)"))

data/raw/snv: present
data/raw/indel: present


## Analysis

In [4]:
# Inventory: every file with size and SHA-256. '._' AppleDouble files and .DS_Store are skipped.
scan = [config.TRUTH, config.REFERENCE, config.DERIVED] + [d for name, d in raw_dirs.items() if raw_present[name]]
inventory = pd.concat([io.file_inventory(d, config.REPO_ROOT) for d in scan], ignore_index=True)
inventory["tier"] = inventory["path"].map(lambda p: "local" if is_local(p) else "committed")
inventory["folder"] = inventory["path"].map(lambda p: "/".join(p.split("/")[:3 if p.startswith("data/raw/") else 2]))
assert inventory["path"].is_unique and not inventory["path"].str.contains(r"/\._").any()
assert inventory["sha256"].str.len().eq(64).all() and (inventory["size_bytes"] >= 0).all()
n_appledouble = sum(n.startswith("._") for d in scan for _, _, names in os.walk(str(d)) for n in names)

committed = inventory[inventory["tier"] == "committed"]
local = inventory[inventory["tier"] == "local"]
summary = inventory.groupby(["tier", "folder"]).agg(n_files=("path", "size"), total_bytes=("size_bytes", "sum")).reset_index()
summary

,tier,folder,n_files,total_bytes
0,committed,data/derived,33,92618138
1,committed,data/reference,72,857323961
2,committed,data/truth,2,1843913
3,local,data/derived,1,1362930
4,local,data/raw/indel,400,178560305
5,local,data/raw/snv,960,3942839136


In [5]:
# Exome BED territory: raw sum of interval lengths (legacy calculate_region_size) vs merged unique length
bed = io.read_bed(str(config.EXOME_BED))
assert bed[["start", "end"]].notna().all().all() and (bed["start"] < bed["end"]).all()
raw_sum_bp = io.region_size(str(config.EXOME_BED))  # overlaps counted twice
assert raw_sum_bp == int((bed["end"] - bed["start"]).sum())
merged_bp = io.unique_region_size(bed)  # legacy calculate_unique_region_size
assert 0 < merged_bp <= raw_sum_bp
territory = pd.DataFrame([{
    "bed": rel(config.EXOME_BED), "n_intervals": len(bed), "n_chromosomes": bed["chrom"].nunique(),
    "raw_sum_bp": raw_sum_bp, "merged_bp": merged_bp, "overlap_bp": raw_sum_bp - merged_bp,
}])
territory

,bed,n_intervals,n_chromosomes,raw_sum_bp,merged_bp,overlap_bp
0,data/reference/sorted_exome_hc.bed.gz,272014,22,80762432,80711709,50723


In [6]:
# Truth VCFs by FILTER value and variant type. FILTER is tested as a token: the SNV file writes
# 'HighConf;PASS' and the indel file 'PASS;HighConf'.
census = pd.concat([io.vcf_census(str(p)).assign(file=rel(p)) for p in (config.TRUTH_VCF, TRUTH_INDEL_VCF)], ignore_index=True)
census = census[["file", "filter", "has_pass", "variant_type", "n_records", "n_multiallelic"]]
assert census["has_pass"].all(), "a truth record without a PASS token"
assert set(census.loc[census["file"] == rel(config.TRUTH_VCF), "variant_type"]) == {"SNV"}
assert set(census.loc[census["file"] == rel(TRUTH_INDEL_VCF), "variant_type"]) <= {"INS", "DEL"}
census

,file,filter,has_pass,variant_type,n_records,n_multiallelic
0,data/truth/hc_bed_filtered.recode.vcf,HighConf;PASS,True,SNV,1089,0
1,data/truth/hc_bed_filtered.recode.vcf,MedConf;PASS,True,SNV,72,0
2,data/truth/hc_bed_filtered_INDELS.recode.vcf,PASS;HighConf,True,DEL,43,0
3,data/truth/hc_bed_filtered_INDELS.recode.vcf,PASS;HighConf,True,INS,2,0
4,data/truth/hc_bed_filtered_INDELS.recode.vcf,PASS;MedConf,True,DEL,2,0
5,data/truth/hc_bed_filtered_INDELS.recode.vcf,PASS;MedConf,True,INS,1,0


## Save tables

In [7]:
cols = ["path", "size_bytes", "sha256"]
io.write_csv(committed[cols], STAGE.path(OUT / "tables" / "inventory_committed.csv"), sort_by="path")
if len(local):
    io.write_csv(local[cols], STAGE.path(OUT / "tables" / "inventory_local.csv"), sort_by="path")
io.write_csv(summary, STAGE.path(OUT / "tables" / "inventory_summary.csv"), sort_by=["tier", "folder"])
io.write_csv(territory, STAGE.path(OUT / "tables" / "bed_territory.csv"), sort_by="bed")
io.write_csv(census, STAGE.path(OUT / "tables" / "truth_census.csv"), sort_by=["file", "filter", "variant_type"])

## Figures

None.

## Parity check

Against the legacy caches in `data/derived/`, using only committed files, so it runs without `data/raw/`:
- the legacy TMB equals variants per run divided by the raw-sum territory, for all 480 runs, in both legacy tables; the merged territory does not reproduce it (D11);
- the truth SNV set reproduces the whole-study rows of `union_metrics.csv` (IoU, precision, recall, F1).

When `data/raw/` is present, the raw file lists are also compared with the legacy file lists. Otherwise those checks are reported as SKIP. Results are moved out of staging only if every check that ran passes.

In [8]:
legacy_meta = pd.read_csv(config.LEGACY_METADATA)
legacy_meta["TestCaseNo"] = legacy_meta["TestCaseNo"].map(io.canon)
legacy_meta = legacy_meta.set_index("TestCaseNo")
legacy_sets = io.load_sets(config.LEGACY_SETS)
legacy_tmb = pd.read_csv(config.LEGACY_TMB)
legacy_tmb["TestCase"] = legacy_tmb["TestCase"].map(io.canon)
legacy_tmb = legacy_tmb.set_index("TestCase")["TMB"]
n_variants = pd.Series({k: len(legacy_sets[k]) for k in legacy_meta.index}).reindex(legacy_meta.index)

parity = audit.Parity()
committed_paths = set(committed["path"])
parity.check("legacy caches are in the committed inventory", all(rel(p) in committed_paths for p in LEGACY_CACHES))
n_match = int((n_variants.values == legacy_meta["Result VCF Count"].values).sum())
parity.check("legacy sets_dict.csv variants per run == legacy Result VCF Count",
             n_match == len(legacy_meta) == EXPECTED_SNV_RUNS, "{}/{} match".format(n_match, len(legacy_meta)))


def tmb_diffs(territory_bp):
    """Max |difference| of variants / (territory / 1e6) from the legacy TMB: vs vcfcomparison_df_full, vs tmb_list_snp."""
    mine = n_variants / (territory_bp / 1e6)
    return (float((mine - legacy_meta["TMB"]).abs().max()), float((mine - legacy_tmb.reindex(mine.index)).abs().max()))


raw_diffs, merged_diffs = tmb_diffs(raw_sum_bp), tmb_diffs(merged_bp)
parity.check("legacy TMB == variants / (raw-sum territory / 1e6), all {} runs, both legacy tables".format(len(n_variants)),
             max(raw_diffs) <= TOLERANCE, "max |diff| {:.2e}".format(max(raw_diffs)))
parity.check("legacy TMB is not reproduced by the merged territory (D11)", min(merged_diffs) > TOLERANCE,
             "max |diff| {:.2e} vs vcfcomparison_df_full, {:.2e} vs tmb_list_snp".format(*merged_diffs))

truth = io.truth_set(str(config.TRUTH_VCF))
n_truth_snv = int(census.loc[census["file"] == rel(config.TRUTH_VCF), "n_records"].sum())
parity.check("truth SNV set size == SNV records of the truth VCF census", len(truth) == n_truth_snv,
             "{} variants".format(len(truth)))
union = pd.read_csv(config.LEGACY_UNION_METRICS, dtype={"TestCase": str})
whole = union[union["Region"].str.lower() == "findings from this study"].copy()
whole["TestCase"] = whole["TestCase"].map(io.canon)
mine = pd.DataFrame({k: (metrics.jaccard(s, truth),) + metrics.prf1(s, truth) for k, s in legacy_sets.items()},
                    index=["Inter/Union", "Precision", "Recall", "f1_score"]).T
worst = max(float(np.abs(whole[c].values - mine.loc[whole["TestCase"], c].values).max()) for c in mine.columns)
parity.check("whole-study rows of union_metrics.csv == legacy sets vs truth set (all {} rows)".format(len(whole)),
             set(whole["TestCase"]) == set(legacy_sets) and worst <= TOLERANCE, "max |diff| {:.2e}".format(worst))


def legacy_paths(csv, prefix):
    df = pd.read_csv(csv)
    return {io.canon(k): v[len(prefix):] if v.startswith(prefix) else v for k, v in zip(df["testcase"], df["filepath"])}


# Raw VCF layout against the legacy file lists: file names only, nothing is parsed
for name, expected, csv, prefix in (("snv", EXPECTED_SNV_RUNS, config.LEGACY_FILENAMES, LEGACY_SNV_ROOT),
                                    ("indel", EXPECTED_INDEL_RUNS, config.LEGACY_FILENAMES_INDELS, LEGACY_INDEL_ROOT)):
    label = "{}: {} run VCFs, one per folder, paths == legacy file list".format(rel(raw_dirs[name]), expected)
    if not raw_present[name]:
        parity.skip(label, "{} is absent".format(rel(raw_dirs[name])))
        continue
    found = {k: os.path.relpath(v, str(raw_dirs[name])) for k, v in io.vcf_filenames(str(raw_dirs[name])).items()}
    n_files = len(io.real_files(str(raw_dirs[name])))
    parity.check(label, len(found) == n_files == expected and found == legacy_paths(csv, prefix),
                 "{} folders, {} VCFs".format(len(found), n_files))

parity.finish(OUT / "audit" / "parity.txt", STAGE)

parity: PASS (8 checks)
PASS  legacy caches are in the committed inventory
PASS  legacy sets_dict.csv variants per run == legacy Result VCF Count  [480/480 match]
PASS  legacy TMB == variants / (raw-sum territory / 1e6), all 480 runs, both legacy tables  [max |diff| 7.11e-15]
PASS  legacy TMB is not reproduced by the merged territory (D11)  [max |diff| 2.57e-02 vs vcfcomparison_df_full, 2.57e-02 vs tmb_list_snp]
PASS  truth SNV set size == SNV records of the truth VCF census  [1161 variants]
PASS  whole-study rows of union_metrics.csv == legacy sets vs truth set (all 1440 rows)  [max |diff| 2.22e-16]
PASS  data/raw/snv: 480 run VCFs, one per folder, paths == legacy file list  [480 folders, 480 VCFs]
PASS  data/raw/indel: 320 run VCFs, one per folder, paths == legacy file list  [320 folders, 320 VCFs]



## Audit summary

In [9]:
overlap_pct = 100.0 * territory.loc[0, "overlap_bp"] / territory.loc[0, "raw_sum_bp"]
census_lines = []
for f, g in census.groupby("file"):
    census_lines.append("  {}: {} records; types {}; FILTER {}".format(
        f, int(g["n_records"].sum()),
        {k: int(v) for k, v in g.groupby("variant_type")["n_records"].sum().items()},
        {k: int(v) for k, v in g.groupby("filter")["n_records"].sum().items()}))
lines = [
    "inventory: {} files, {:,} bytes; SHA-256 in tables/inventory_committed.csv{}".format(
        len(inventory), int(inventory["size_bytes"].sum()), " and tables/inventory_local.csv" if len(local) else ""),
    summary.to_string(index=False),
    "data/raw: " + ", ".join("{} {}".format(rel(d), "present" if raw_present[n] else "ABSENT") for n, d in raw_dirs.items()),
    "AppleDouble '._' files skipped: {}".format(n_appledouble),
    "",
    "exome BED {}: {:,} intervals on {} chromosomes".format(
        territory.loc[0, "bed"], territory.loc[0, "n_intervals"], territory.loc[0, "n_chromosomes"]),
    "  raw sum {:,} bp; merged unique {:,} bp; overlap {:,} bp ({:.3f}% of the raw sum)".format(
        territory.loc[0, "raw_sum_bp"], territory.loc[0, "merged_bp"], territory.loc[0, "overlap_bp"], overlap_pct),
    "  legacy TMB divides by the raw sum (max |diff| {:.1e}); the merged length gives max |diff| {:.1e} (D11, undecided)".format(
        max(raw_diffs), min(merged_diffs)),
    "",
    "truth VCFs (FILTER tested as a token; the two files order it differently):",
] + census_lines + [
    "",
    "python {} | pandas {} | numpy {}".format(sys.version.split()[0], pd.__version__, np.__version__),
]
(OUT / "audit").mkdir(parents=True, exist_ok=True)
(OUT / "audit" / "summary.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

inventory: 1468 files, 5,074,548,383 bytes; SHA-256 in tables/inventory_committed.csv and tables/inventory_local.csv
     tier         folder  n_files  total_bytes
committed   data/derived       33     92618138
committed data/reference       72    857323961
committed     data/truth        2      1843913
    local   data/derived        1      1362930
    local data/raw/indel      400    178560305
    local   data/raw/snv      960   3942839136
data/raw: data/raw/snv present, data/raw/indel present
AppleDouble '._' files skipped: 0

exome BED data/reference/sorted_exome_hc.bed.gz: 272,014 intervals on 22 chromosomes
  raw sum 80,762,432 bp; merged unique 80,711,709 bp; overlap 50,723 bp (0.063% of the raw sum)
  legacy TMB divides by the raw sum (max |diff| 7.1e-15); the merged length gives max |diff| 2.6e-02 (D11, undecided)

truth VCFs (FILTER tested as a token; the two files order it differently):
  data/truth/hc_bed_filtered.recode.vcf: 1161 records; types {'SNV': 1161}; FILTER {'High